# AI 201 — Week 4 Demo: Discord Content Filtering Bot

We're building the moderation system for a Kendrick Lamar fan server — a multi-layered safety system that handles everything from obvious hate speech to coordinated raids to users trying to jailbreak the bot.

**The layers we're building (in order — cheapest first):**
1. **Rate limiter** — pre-built, stops volume abuse before checking content
2. **Input validation** — ✍️ we write this: fast keyword checks, no LLM call
3. **LLM content filter** — ✍️ we write this: handles the edge cases the keyword layer can't
4. **Prompt injection defense** — ✍️ we write this: catches users trying to override the bot

**What you write live:** the community rules, all four layers (input validation, LLM filter, injection defense), and the function that wires them together.

In [ ]:
# Cell 1 — Setup
from dotenv import load_dotenv
load_dotenv()

import sys, json, time
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve()))

from data.test_messages import ALL_SCENARIOS, print_scenario
from layers import RateLimiter, ModerationLogger
from utils import LLMClient

llm     = LLMClient()
limiter = RateLimiter(user_limit=5, server_limit=20, window_seconds=10)
log     = ModerationLogger()

print('Ready.')
print(f'LLM: {llm.model}')
print(f'Rate limiter: {limiter.user_limit} msg/user, {limiter.server_limit} msg/server per {limiter.window_seconds}s')
print(f'Scenarios loaded: {list(ALL_SCENARIOS.keys())}')

---
## Part 1: Preview the Test Messages

Before we build anything, look at the messages we'll be running through the system. Notice that some are obvious, some are genuinely hard to call, and some are trying to break the bot entirely.

In [ ]:
# Preview each scenario — run this cell, then scroll through
for scenario in ALL_SCENARIOS:
    print_scenario(scenario)

---
## Part 2: Write the Community Rules

Content filtering is a design decision. The rules encode what *this community* considers acceptable — which is different from what any other community would say.

**✍️ LIVE CODE — write the rules together:**

In [ ]:
# -- INSTRUCTOR: Write this live. ---------------------------------------------
#
# Ask students before writing: "What should earn a ban vs. a timeout vs. a pass
# in an academic discussion community?"
#
# Take a few answers. Then write the rules dict together.
#
# Key talking point:
#   "Notice these are community-specific norms. A policy for a course forum
#    will differ from a gaming server or a fan community."
#
# After writing: ask students, "What would change if this were a professional
# workplace chat, a public social platform, or a K-12 class community?"
# -----------------------------------------------------------------------------

COMMUNITY_RULES = {
    "server_name": "AI 201 Course Discussion Server",
    "vibe": "Respectful, evidence-based discussion. Disagreement is welcome; harassment is not.",
    "ban_immediately": [
        "Hate speech, slurs, or discriminatory harassment",
        "Doxxing or sharing private personal information",
        "Credible threats of violence or harm",
        "Posting illegal content",
    ],
    "timeout_30_min": [
        "Targeted personal attacks or repeated hostility",
        "Deliberately disruptive bad-faith behavior",
        "Spam, flooding, or repeated off-topic posting",
        "Attempts to bypass moderation rules",
    ],
    "allow": [
        "Respectful critique of ideas, code, or arguments",
        "Strong opinions supported by reasoning",
        "Constructive disagreement and debate",
        "Informal language that is not abusive or targeted",
    ],
}

print(f"Rules written for: {COMMUNITY_RULES['server_name']}")
print(f"  Ban offenses:     {len(COMMUNITY_RULES['ban_immediately'])}")
print(f"  Timeout offenses: {len(COMMUNITY_RULES['timeout_30_min'])}")
print(f"  Allowed:          {len(COMMUNITY_RULES['allow'])}")

---
## Part 3: Write the Input Validation Layer

The cheapest layer runs first. No LLM call — just fast checks that catch obvious violations and basic abuse patterns before we spend any compute.

**✍️ LIVE CODE — write this together:**

In [ ]:
# ── INSTRUCTOR: Write this live. ────────────────────────────────────────────
#
# This layer does three things:
#   1. Rate limit check (delegates to the pre-built RateLimiter)
#   2. Length check (very long messages = spam or abuse)
#   3. Keyword scan (slurs, threats, spam patterns — instant ban/timeout)
#
# Ask students before writing: "What can we check WITHOUT an LLM?"
# Take answers. Then land on: length, keywords, frequency.
#
# Ask while writing the keyword list: "Why don't we just keyword-filter everything?"
#   → Because 'hurt' in 'this song HURTS' is fine.
#   → Because 'kill' in 'Kendrick killed that verse' is fine.
#   → Context matters — which is exactly why we need the LLM layer next.
# ────────────────────────────────────────────────────────────────────────────

# Words that warrant instant action regardless of context
BAN_KEYWORDS   = ["[REDACTED SLUR]"]  # extend with actual slurs — omitted here for demo
TIMEOUT_KEYWORDS = ["i will find you", "i know where you live", "posting your address"]
SPAM_PATTERNS    = ["buy followers", "click here", "http://spam"]  # extend as needed


def validate_input(user_id: str, username: str, content: str) -> dict | None:
    """
    Run fast, cheap checks before sending a message to the LLM.

    Returns a decision dict if this layer catches something,
    or None if the message should continue to the next layer.
    """
    content_lower = content.lower()

    # 1. Rate limit check
    rate_result = limiter.check(user_id)
    if not rate_result["allowed"]:
        return {
            "decision": "rate_limited",
            "reason": rate_result["reason"],
            "layer": "rate_limiter",
            "confidence": "high",
        }

    # 2. Length check
    if len(content) > 2000:
        return {
            "decision": "timeout",
            "reason": f"Message too long ({len(content)} chars) — likely spam",
            "layer": "input_validation",
            "confidence": "high",
        }

    # 3. Keyword scan — ban
    for keyword in BAN_KEYWORDS:
        if keyword.lower() in content_lower:
            return {
                "decision": "ban",
                "reason": f"Hate speech detected (keyword: '{keyword}')",
                "layer": "input_validation",
                "confidence": "high",
            }

    # 4. Keyword scan — timeout
    for keyword in TIMEOUT_KEYWORDS + SPAM_PATTERNS:
        if keyword.lower() in content_lower:
            return {
                "decision": "timeout",
                "reason": f"Flagged pattern detected ('{keyword}')",
                "layer": "input_validation",
                "confidence": "high",
            }

    # Nothing caught — pass to the next layer
    return None


print('validate_input() defined.')
print(f'  Ban keywords:     {len(BAN_KEYWORDS)}')
print(f'  Timeout keywords: {len(TIMEOUT_KEYWORDS)}')
print(f'  Spam patterns:    {len(SPAM_PATTERNS)}')

In [ ]:
# ── BEFORE YOU LIVE-CODE THE LLM FILTER: show raw output + parsing ─────────
#
# Students are about to implement the LLM filter themselves in the lab.
# Before writing the function, show them what the raw API response looks like
# and why parsing + validation can't be skipped.
#
# Run this cell, then walk through the output top to bottom.
# ─────────────────────────────────────────────────────────────────────────────

# Step 1: Make a raw call with no parsing — show the actual response object
raw_response = llm.client.chat.completions.create(
    model=llm.model,
    messages=[
        {"role": "system", "content": "Classify this message. Return ONLY the word: allow, timeout, or ban."},
        {"role": "user", "content": "I hate how good that ending was"},
    ],
)
raw_text = raw_response.choices[0].message.content
print('=== RAW LLM OUTPUT (no parsing) ===')
print(repr(raw_text))   # repr() shows whitespace, quotes, and casing exactly
print()

# Step 2: Show what can go wrong with the raw value
VALID_DECISIONS = ['allow', 'timeout', 'ban']
print('=== NAIVE VALIDATION (breaks on casing/whitespace) ===')
print(f'raw_text in VALID_DECISIONS: {raw_text in VALID_DECISIONS}')
print()

# Step 3: Normalize-before-validate
normalized = raw_text.strip().lower()
print('=== NORMALIZE THEN VALIDATE ===')
print(f'normalized: {repr(normalized)}')
print(f'normalized in VALID_DECISIONS: {normalized in VALID_DECISIONS}')
print()

# ── Ask students ─────────────────────────────────────────────────────────────
# "What are three ways the raw output could still fail validation even after
#  .strip().lower()?"
#   → 'Allow.' with a period
#   → '"allow"' wrapped in quotes
#   → 'allow: this message is safe' — extra content after the word
# "What should the function return if normalization still can't produce a
#  valid decision?" → the fallback is a design decision, not a default.
print('--- Instructor bridge: this is why the lab asks you to normalize AND validate ---')

---
## Part 4: Write the LLM Content Filter

For messages that pass input validation — the ambiguous ones — we ask the LLM. It reads the community rules and returns a structured verdict.

**✍️ LIVE CODE — write the system prompt and filter function:**

In [ ]:
# ── INSTRUCTOR: Write this live. ────────────────────────────────────────────
#
# Two things to write:
#   1. The system prompt (formats the community rules into instructions)
#   2. The llm_filter() function (calls llm.moderate() and records the decision)
#
# Key talking point on the system prompt:
#   "The system prompt IS the safety policy. Every word matters.
#    Vague instructions → inconsistent decisions.
#    Specific rules + examples → much more predictable behavior."
#
# Ask after writing: "What would happen if we put the system prompt
#   in the user turn instead of the system turn?"
#   → The model gives it less weight. System prompts have higher authority.
#   → Also: users could try to override it. Which leads to Part 5.
# ────────────────────────────────────────────────────────────────────────────

def build_system_prompt(rules: dict) -> str:
    """Format the community rules dict into a moderator system prompt."""
    ban_rules     = "\n".join(f"  - {r}" for r in rules["ban_immediately"])
    timeout_rules = "\n".join(f"  - {r}" for r in rules["timeout_30_min"])
    allow_rules   = "\n".join(f"  - {r}" for r in rules["allow"])

    return f"""You are the moderation bot for {rules['server_name']}.
Server vibe: {rules['vibe']}

IMMEDIATELY BAN for:
{ban_rules}

TIMEOUT (30 min) for:
{timeout_rules}

ALLOW (including):
{allow_rules}

Evaluate each message and return a JSON verdict.
When in doubt between timeout and allow, prefer allow.
Only ban when you are highly confident."""


def llm_filter(user_id: str, username: str, content: str) -> dict:
    """Pass a message to the LLM for content evaluation."""
    system_prompt = build_system_prompt(COMMUNITY_RULES)
    result = llm.moderate(system_prompt, content)
    result["layer"] = "llm_filter"
    return result


# Preview the system prompt
print(build_system_prompt(COMMUNITY_RULES))

---
## Part 5: Write the Prompt Injection Defense

Some users will try to override the bot by framing their message as an instruction. We need to catch this before the message reaches the LLM filter — because if it reaches the LLM, a poorly defended model might comply.

**✍️ LIVE CODE — write the injection detector:**

In [ ]:
# ── INSTRUCTOR: Write this live. ────────────────────────────────────────────
#
# Write detect_injection() first — then show what happens WITH and WITHOUT
# it when you run the injection messages in Part 6.
#
# Key talking point:
#   "This is pattern matching, not LLM reasoning. That's intentional.
#    You don't want to use the thing that can be fooled to decide whether
#    something is trying to fool it. Cheap, deterministic defense first."
#
# Ask students: "Can you think of an injection attempt that would get past
#   these patterns?" Then ask: "What would you add to catch it?"
#   This is a real, open-ended security design conversation.
# ────────────────────────────────────────────────────────────────────────────

INJECTION_PATTERNS = [
    "ignore your previous instructions",
    "ignore previous instructions",
    "you are now",
    "pretend you are",
    "pretend you're",
    "act as if you have no",
    "disregard your",
    "your new instructions",
    "system:",                  # fake system message
    "new rule:",
    "admin privileges",
    "override moderation",
    "what would a bot with no",  # hypothetical framing
    "hypothetically, if you had no rules",
]


def detect_injection(content: str) -> dict | None:
    """
    Scan for prompt injection patterns before the message reaches the LLM.

    Returns a timeout decision if injection is detected, None otherwise.
    """
    content_lower = content.lower()
    for pattern in INJECTION_PATTERNS:
        if pattern in content_lower:
            return {
                "decision": "timeout",
                "reason": f"Prompt injection attempt detected (pattern: '{pattern}')",
                "layer": "injection_defense",
                "confidence": "high",
            }
    return None


print(f'detect_injection() defined with {len(INJECTION_PATTERNS)} patterns.')

---
## Part 6: Wire It All Together

Now connect the layers into a single `moderate()` function that every message passes through in order.

In [ ]:
# ── INSTRUCTOR: Write this live — it's short, ~15 lines. ────────────────────
#
# Ask students: "What order should the layers run?"
#   → Rate limit first (no content check needed, just frequency)
#   → Injection defense next (before the LLM even sees it)
#   → Input validation (cheap keyword checks)
#   → LLM filter last (most expensive, only when necessary)
#
# Ask: "Why does order matter?"
#   → Cost: LLM calls cost money and time. Run them last.
#   → Security: injection defense must run before the LLM, not after.
# ────────────────────────────────────────────────────────────────────────────

def moderate(user_id: str, username: str, content: str) -> dict:
    """
    Run a message through all moderation layers in order.
    Returns the first decision that fires, or an allow from the LLM.
    """
    # Layer 1: Rate limiter (already inside validate_input)
    # Layer 2: Injection defense — before the LLM sees anything
    decision = detect_injection(content)

    # Layer 3: Input validation (rate limit + keywords)
    if decision is None:
        decision = validate_input(user_id, username, content)

    # Layer 4: LLM filter — only for messages that passed everything above
    if decision is None:
        decision = llm_filter(user_id, username, content)

    # Log the decision
    log.log(
        user_id=user_id,
        username=username,
        message=content,
        decision=decision["decision"],
        reason=decision["reason"],
        layer=decision["layer"],
        confidence=decision.get("confidence", ""),
    )
    return decision


print('moderate() defined — all four layers wired together.')

---
## Part 7: Run the Demo Scenarios

In [ ]:
# ── DEMO MOMENT 1 — Clean messages ─────────────────────────────────────────
# Ask students: "Which layer do you expect to handle these?"
# Answer: LLM filter (they pass keyword + injection checks)
# Ask after: "Was it worth an LLM call for these? What's the alternative?"
# ───────────────────────────────────────────────────────────────────────────

print('=== SCENARIO 1: Clean messages ===\n')
for msg in ALL_SCENARIOS['clean']:
    result = moderate(msg.user_id, msg.username, msg.content)
    print(f'[{msg.username}]')
    print(f'  Message:  "{msg.content[:80]}..."')
    print(f'  Decision: {result["decision"]} ({result["layer"]}) — {result["reason"]}')
    print()

In [ ]:
# ── DEMO MOMENT 2 — Obvious violations ─────────────────────────────────────
# Ask: "Which layer catches these — and why does that matter for cost?"
# These should all be caught by input_validation, no LLM call needed.
# ───────────────────────────────────────────────────────────────────────────

print('=== SCENARIO 2: Obvious violations ===\n')
for msg in ALL_SCENARIOS['obvious_violation']:
    result = moderate(msg.user_id, msg.username, msg.content)
    print(f'[{msg.username}]')
    print(f'  Message:  "{msg.content[:80]}"')
    print(f'  Decision: {result["decision"]} ({result["layer"]}) — {result["reason"]}')
    print()

In [ ]:
# ── DEMO MOMENT 3 — Edge cases ──────────────────────────────────────────────
# These are the genuinely hard ones. The keyword layer can't handle them.
# Ask BEFORE running: "What would you decide for each of these?"
# Then run them and compare students' instincts to the model's decisions.
# Ask: "Do you agree? Where did it surprise you?"
# ───────────────────────────────────────────────────────────────────────────

print('=== SCENARIO 3: Edge cases (LLM filter) ===\n')
for msg in ALL_SCENARIOS['edge_case']:
    result = moderate(msg.user_id, msg.username, msg.content)
    print(f'[{msg.username}]')
    print(f'  Message:  "{msg.content[:100]}"')
    print(f'  Note:     {msg.notes}')
    print(f'  Decision: {result["decision"]} ({result["layer"]}, {result.get("confidence","")} confidence)')
    print(f'  Reason:   {result["reason"]}')
    print()

In [ ]:
# ── DEMO MOMENT 4 — Prompt injection ───────────────────────────────────────
# First: run ONE injection message with detect_injection() commented out
# in moderate() to show what happens WITHOUT the defense.
# Then: restore it and show the defense catching it.
#
# Ask: "Why can't we just tell the LLM to ignore injection attempts?"
#   → Because we're asking the thing that can be fooled to protect itself.
#   → Pattern matching is deterministic. The LLM is not.
# ───────────────────────────────────────────────────────────────────────────

print('=== SCENARIO 4: Prompt injection attempts ===\n')
for msg in ALL_SCENARIOS['injection']:
    result = moderate(msg.user_id, msg.username, msg.content)
    print(f'[{msg.username}]')
    print(f'  Message:  "{msg.content[:100]}"')
    print(f'  Decision: {result["decision"]} ({result["layer"]}) — {result["reason"]}')
    print()

In [ ]:
# ── DEMO MOMENT 5 — Raid simulation ────────────────────────────────────────
# 15 different users sending messages in quick succession.
# The rate limiter fires at the server level — not per-user.
# Ask: "Why do we need a server-wide limit, not just per-user?"
#   → A coordinated raid uses many accounts. Per-user limits alone won't catch it.
# ───────────────────────────────────────────────────────────────────────────

print('=== SCENARIO 5: Raid simulation (15 users) ===\n')
limiter.reset()   # clear any state from previous scenarios

blocked_count = 0
for msg in ALL_SCENARIOS['raid']:
    result = moderate(msg.user_id, msg.username, msg.content)
    status = '🚫 BLOCKED' if result['decision'] == 'rate_limited' else '  passed'
    if result['decision'] == 'rate_limited':
        blocked_count += 1
    print(f'{status}  [{msg.username}]: "{msg.content[:60]}"')

print(f'\n{blocked_count}/{len(ALL_SCENARIOS["raid"])} raid messages blocked by rate limiter.')

In [ ]:
# ── BEFORE REVIEWING THE LOG: show how JSONL writing actually works ──────────
#
# The ModerationLogger is pre-built. Before we read the log, open the hood
# and show students the write pattern they'll implement themselves in the lab.
#
# ─────────────────────────────────────────────────────────────────────────────

import json as _json   # already imported, renaming to avoid collision in this cell
from pathlib import Path

# A JSONL record: one Python dict, one line in the file
sample_record = {
    'timestamp': '2025-11-01T14:22:01',
    'user_id': 'user_42',
    'decision': 'timeout',
    'reason': 'Targeted harassment of another user',
    'layer': 'llm_filter',
    'message_preview': 'I hate [user] so much, they should just...',
}

# ── CORRECT: json.dumps() + newline — one record, one line ──────────────────
correct_line = _json.dumps(sample_record) + '\n'
print('=== CORRECT (JSONL) ===')
print(correct_line)

# ── WRONG: json.dump() with indent — breaks the format ──────────────────────
import io
buf = io.StringIO()
_json.dump(sample_record, buf, indent=2)
print('=== WRONG (multi-line JSON, breaks JSONL parsers) ===')
print(buf.getvalue())
print()

# ── The actual write pattern ─────────────────────────────────────────────────
print('=== HOW TO WRITE IT ===')
print('''
# Create directory if it doesn't exist
Path('logs').mkdir(exist_ok=True)

# Append one record — 'a' mode, never 'w' (which overwrites)
with open('logs/audit.jsonl', 'a') as f:
    f.write(json.dumps(record) + '\\n')
''')

# ── Ask students ─────────────────────────────────────────────────────────────
# "Why append mode instead of write mode?"
#   → Write mode overwrites the file. Every run would erase the history.
# "Why does the multi-line format break JSONL tools?"
#   → Splunk, Datadog, CloudWatch all expect one record per line.
#     Multi-line pretty-print means one 'record' spans many lines — the parser
#     sees garbage after the first.
# "What happens if two processes write to the log at the same time?"
#   → Good answer: at scale, you'd use a log aggregation service, not a flat file.
print('--- Instructor bridge: this is the exact pattern the lab asks you to implement in auditor.py ---')

---
## Part 8: Review the Moderation Log

In [ ]:
# ── DEMO MOMENT 6 — Show the full log ──────────────────────────────────────
# Ask: "Why does every decision get logged, even the allows?"
#   → You need the full picture to calibrate your rules.
#   → If 90% of messages are being allowed, are your rules too loose?
#   → If 40% are being timed out, they're probably too strict.
#   → The log is how you find out — not by guessing.
# ───────────────────────────────────────────────────────────────────────────

print(f'Total moderation decisions logged: {len(log)}\n')
log.print_log()

print('Summary:')
for decision, count in sorted(log.summary().items()):
    print(f'  {decision:<14}: {count}')